
# UsedCarAI — 01 Data Cleaning & Dataset Preparation




In [ ]:
from pathlib import Path
import os
import sys

# Discover CPE-306 when opened from the project or UsedcarAi directory.
# Set USEDCAR_PROJECT_DIR to use a different checkout location.
_candidates = (Path.cwd(), *Path.cwd().parents)
_default_project = next(
    (p for p in _candidates if (p / "dataset").is_dir() and (p / "app").is_dir()),
    Path("/home/regen/Works/CPE-306"),
)
PROJECT_DIR = Path(os.environ.get("USEDCAR_PROJECT_DIR", _default_project)).expanduser().resolve()
DATASET_DIR = Path(os.environ.get("DATASET_DIR", PROJECT_DIR / "dataset")).expanduser().resolve()
if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")
print("Python:", sys.executable)
print("Dataset:", DATASET_DIR)
print("Use the Python kernel from the project's ML environment.")


In [ ]:

from pathlib import Path
import json
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

PROJECT_ROOT = DATASET_DIR
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CLEANED_DIR = PROJECT_ROOT / "data" / "cleaned"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR = RESULTS_DIR / "figures"

for d in [RAW_DIR, CLEANED_DIR, PROCESSED_DIR, METRICS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ชื่อไฟล์ที่แนะนำ
RAW_FILE = RAW_DIR / "one2car_thailand_20260822.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_FILE:", RAW_FILE)
print("Raw file exists:", RAW_FILE.exists())

if not RAW_FILE.exists():
    print("\n[ERROR] ไม่พบไฟล์ CSV")
    print("กรุณานำไฟล์ one2car_thailand_20260822.csv ไปไว้ที่:")
    print(RAW_FILE)
    raise FileNotFoundError(RAW_FILE)


## Step 1 — โหลดและตรวจสอบ Schema

In [ ]:

df_raw = pd.read_csv(RAW_FILE)

print("Shape:", df_raw.shape)
print("\nColumns:")
for c in df_raw.columns:
    print(" -", c)

display(df_raw.head(3))


In [ ]:

EXPECTED_COLUMNS = [
    "listing_id", "source_url", "title", "brand", "model", "sub_model",
    "model_year", "raw_price", "price", "raw_mileage", "mileage",
    "fuel_type", "transmission", "engine_size", "body_type", "color",
    "province", "location", "seller_name", "seller_type", "dealer_slug",
    "seller_url", "number_of_seats", "image_url", "description",
    "scraped_at"
]

missing_columns = [c for c in EXPECTED_COLUMNS if c not in df_raw.columns]
extra_columns = [c for c in df_raw.columns if c not in EXPECTED_COLUMNS]

print("Missing expected columns:", missing_columns)
print("Extra columns:", extra_columns)

if missing_columns:
    raise ValueError(
        f"Dataset schema ไม่ตรงกับที่กำหนด ขาดคอลัมน์: {missing_columns}"
    )


## Step 2 — Data Quality Audit

In [ ]:

quality = pd.DataFrame({
    "column": df_raw.columns,
    "dtype": [str(df_raw[c].dtype) for c in df_raw.columns],
    "missing_count": [int(df_raw[c].isna().sum()) for c in df_raw.columns],
    "missing_rate": [float(df_raw[c].isna().mean()) for c in df_raw.columns],
    "unique_count": [int(df_raw[c].nunique(dropna=True)) for c in df_raw.columns],
})

display(quality.sort_values("missing_rate", ascending=False))

quality.to_csv(METRICS_DIR / "01_raw_data_quality.csv", index=False, encoding="utf-8-sig")


In [ ]:

print("Exact duplicate rows:", int(df_raw.duplicated().sum()))
print("Duplicate listing_id:", int(df_raw["listing_id"].duplicated().sum()))
print("Missing seller_name:", int(df_raw["seller_name"].isna().sum()), "/", len(df_raw))

print("\nTarget summary:")
display(df_raw["price"].describe())

print("\nModel year range:", df_raw["model_year"].min(), "to", df_raw["model_year"].max())
print("Mileage range:", df_raw["mileage"].min(), "to", df_raw["mileage"].max())



## Step 3 — Cleaning Rules

กฎในขั้นตอนนี้เน้น **ลบเฉพาะค่าที่ผิดรูปแบบหรือเป็นไปไม่ได้อย่างชัดเจน** และเก็บค่าราคา/เลขไมล์ที่สูงมากไว้เพื่อการตรวจสอบภายหลัง

### ค่าที่จะถูกตัดออก
- `listing_id` ต้องมีและต้องไม่ซ้ำ
- `price` ต้องเป็นตัวเลขและ `> 0`
- `mileage` ต้องเป็นตัวเลขและ `>= 0`
- `model_year` ต้องอยู่ในช่วง 1950–ปีปัจจุบันของ Dataset
- `engine_size` ถ้ามีค่าต้อง `> 0`
- `number_of_seats` ถ้ามีค่าต้องอยู่ในช่วง 1–20
- ลบ exact duplicate rows

### ค่าที่จะ **ไม่ถูกลบอัตโนมัติ**
- รถราคาสูง
- รถเลขไมล์สูง
- รถคลาสสิก
- หมวดหมู่ที่พบจำนวนน้อย

เหตุผลคือค่าดังกล่าวอาจเป็นรถจริง และไม่ควรใช้ threshold แบบตลาดสหรัฐฯ หรือแปลงหน่วย USD ใน Dataset ไทย


In [ ]:

df = df_raw.copy()

# Normalize string columns
string_cols = [
    "title", "brand", "model", "sub_model", "raw_price", "raw_mileage",
    "fuel_type", "transmission", "body_type", "color", "province",
    "location", "seller_name", "seller_type", "dealer_slug", "seller_url",
    "image_url", "description", "source_url"
]

for c in string_cols:
    if c in df.columns:
        df[c] = df[c].astype("string").str.strip()
        df[c] = df[c].replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})

# Numeric coercion
numeric_cols = ["listing_id", "model_year", "price", "mileage", "engine_size", "number_of_seats"]
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Timestamp
df["scraped_at"] = pd.to_datetime(df["scraped_at"], errors="coerce")

initial_rows = len(df)

# Remove exact duplicate rows
df = df.drop_duplicates()

# Remove duplicate listing_id, keeping the first row
df = df.dropna(subset=["listing_id"])
df = df.drop_duplicates(subset=["listing_id"], keep="first")

# Hard validity rules only
current_year = int(df["scraped_at"].dt.year.dropna().mode().iloc[0]) if df["scraped_at"].notna().any() else 2026

valid_mask = (
    df["price"].notna() & (df["price"] > 0) &
    df["mileage"].notna() & (df["mileage"] >= 0) &
    df["model_year"].notna() & (df["model_year"] >= 1950) & (df["model_year"] <= current_year) &
    (df["engine_size"].isna() | (df["engine_size"] > 0)) &
    (df["number_of_seats"].isna() | df["number_of_seats"].between(1, 20))
)

invalid_rows = int((~valid_mask).sum())
df = df.loc[valid_mask].copy()

# Columns that are not useful as predictive features but can remain as traceability data.
# raw_price/raw_mileage are retained in raw data but removed from the cleaned modeling dataset
# because parsed numeric columns already exist.
drop_clean_cols = ["raw_price", "raw_mileage", "seller_name"]
clean_df = df.drop(columns=[c for c in drop_clean_cols if c in df.columns]).copy()

print("Rows before cleaning:", initial_rows)
print("Rows removed:", initial_rows - len(clean_df))
print("Rows after cleaning:", len(clean_df))


## Step 4 — Flag Extreme Values (ไม่ลบทิ้งอัตโนมัติ)

In [ ]:

# Robust flags for review; these rows are kept.
price_q1, price_q3 = clean_df["price"].quantile([0.25, 0.75])
price_iqr = price_q3 - price_q1
price_upper_iqr = price_q3 + 1.5 * price_iqr

mileage_q1, mileage_q3 = clean_df["mileage"].quantile([0.25, 0.75])
mileage_iqr = mileage_q3 - mileage_q1
mileage_upper_iqr = mileage_q3 + 1.5 * mileage_iqr

clean_df["price_extreme_flag"] = clean_df["price"] > price_upper_iqr
clean_df["mileage_extreme_flag"] = clean_df["mileage"] > mileage_upper_iqr

extreme_summary = pd.DataFrame({
    "flag": ["price_extreme_flag", "mileage_extreme_flag"],
    "count": [
        int(clean_df["price_extreme_flag"].sum()),
        int(clean_df["mileage_extreme_flag"].sum())
    ],
    "rate": [
        float(clean_df["price_extreme_flag"].mean()),
        float(clean_df["mileage_extreme_flag"].mean())
    ]
})

display(extreme_summary)
extreme_summary.to_csv(METRICS_DIR / "02_extreme_value_flags.csv", index=False, encoding="utf-8-sig")


In [ ]:

# Save cleaned data first, including traceability flags.
clean_path = CLEANED_DIR / "one2car_cleaned.csv"
clean_df.to_csv(clean_path, index=False, encoding="utf-8-sig")

print("Saved:", clean_path)



## Step 5 — Feature Engineering

สร้างเฉพาะ Feature ที่ไม่ใช้ Target (`price`) เป็น input

- `vehicle_age`
- `vehicle_age_sq`
- `mileage_log1p`
- `mileage_per_year`

Feature ที่มีความเสี่ยงต่อ leakage เช่น `price` หรือข้อมูลที่สร้างจากราคา จะไม่ถูกใช้เป็น input


In [ ]:

fe_df = clean_df.copy()

# Use scrape year from the same row when available.
scrape_year = fe_df["scraped_at"].dt.year.fillna(current_year).astype(int)

fe_df["vehicle_age"] = (scrape_year - fe_df["model_year"]).clip(lower=0)
fe_df["vehicle_age_sq"] = fe_df["vehicle_age"] ** 2
fe_df["mileage_log1p"] = np.log1p(fe_df["mileage"])

# Avoid division by zero and unrealistic tiny age.
age_for_rate = fe_df["vehicle_age"].clip(lower=1)
fe_df["mileage_per_year"] = fe_df["mileage"] / age_for_rate

# Keep the raw flag columns for auditability, but they should NOT be used as model features by default.
print(fe_df[[
    "brand", "model", "sub_model", "model_year", "price", "mileage",
    "vehicle_age", "vehicle_age_sq", "mileage_log1p", "mileage_per_year"
]].head())


## Step 6 — Train / Validation / Test Split (70 / 10 / 20)

In [ ]:

from sklearn.model_selection import train_test_split

# Stratify by price decile so all 3 models see comparable price distributions.
# qcut with duplicates='drop' prevents errors if many identical prices occur.
price_band = pd.qcut(fe_df["price"], q=10, labels=False, duplicates="drop")

train_val, test = train_test_split(
    fe_df,
    test_size=0.20,
    random_state=42,
    stratify=price_band
)

train_price_band = pd.qcut(train_val["price"], q=10, labels=False, duplicates="drop")

train, validation = train_test_split(
    train_val,
    test_size=0.125,  # 10% of total = 12.5% of remaining 80%
    random_state=42,
    stratify=train_price_band
)

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

assert abs(len(train)/len(fe_df) - 0.70) < 0.01
assert abs(len(validation)/len(fe_df) - 0.10) < 0.01
assert abs(len(test)/len(fe_df) - 0.20) < 0.01


In [ ]:

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train), len(validation), len(test)],
    "share": [len(train)/len(fe_df), len(validation)/len(fe_df), len(test)/len(fe_df)],
    "price_median": [train["price"].median(), validation["price"].median(), test["price"].median()],
    "price_mean": [train["price"].mean(), validation["price"].mean(), test["price"].mean()],
})

display(split_summary)
split_summary.to_csv(METRICS_DIR / "03_split_summary.csv", index=False, encoding="utf-8-sig")

train.to_csv(PROCESSED_DIR / "train.csv", index=False, encoding="utf-8-sig")
validation.to_csv(PROCESSED_DIR / "validation.csv", index=False, encoding="utf-8-sig")
test.to_csv(PROCESSED_DIR / "test.csv", index=False, encoding="utf-8-sig")

print("Processed files saved:")
print(" -", PROCESSED_DIR / "train.csv")
print(" -", PROCESSED_DIR / "validation.csv")
print(" -", PROCESSED_DIR / "test.csv")


## Step 7 — บันทึก Data Dictionary / Cleaning Summary

In [ ]:

data_dictionary = pd.DataFrame({
    "feature": [
        "brand", "model", "sub_model", "model_year", "price", "mileage",
        "fuel_type", "transmission", "engine_size", "body_type", "color",
        "province", "location", "seller_type", "number_of_seats",
        "vehicle_age", "vehicle_age_sq", "mileage_log1p", "mileage_per_year"
    ],
    "role": [
        "categorical input", "categorical input", "categorical input", "numeric input",
        "target", "numeric input", "categorical input", "categorical input",
        "numeric input", "categorical input", "categorical input",
        "categorical input", "categorical input", "categorical input",
        "numeric input", "engineered input", "engineered input",
        "engineered input", "engineered input"
    ]
})

data_dictionary.to_csv(
    METRICS_DIR / "04_model_feature_dictionary.csv",
    index=False,
    encoding="utf-8-sig"
)

cleaning_summary = pd.DataFrame({
    "item": [
        "raw_rows", "clean_rows", "rows_removed",
        "exact_duplicates_before_cleaning",
        "duplicate_listing_id_before_cleaning",
        "price_extreme_rows_kept",
        "mileage_extreme_rows_kept"
    ],
    "value": [
        initial_rows,
        len(clean_df),
        initial_rows - len(clean_df),
        int(df_raw.duplicated().sum()),
        int(df_raw["listing_id"].duplicated().sum()),
        int(clean_df["price_extreme_flag"].sum()),
        int(clean_df["mileage_extreme_flag"].sum())
    ]
})

cleaning_summary.to_csv(
    METRICS_DIR / "05_cleaning_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved audit files in:", METRICS_DIR)


## Step 8 — ตรวจสอบผลลัพธ์ขั้นสุดท้าย

In [ ]:

# Missing-value report for cleaned dataset
final_missing = (
    clean_df.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "column"})
)

display(final_missing.head(15))
final_missing.to_csv(
    METRICS_DIR / "06_cleaned_missing_rates.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save missing-rate figure
plt.figure(figsize=(10, 5))
top_missing = final_missing.head(15).sort_values("missing_rate")
plt.barh(top_missing["column"], top_missing["missing_rate"])
plt.xlabel("Missing rate")
plt.title("One2Car Cleaned Dataset — Missing Rate")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "missing_rates.png", dpi=180)
plt.show()

# Save price distribution
plt.figure(figsize=(10, 5))
plt.hist(clean_df["price"], bins=80)
plt.xlabel("Listing price (THB)")
plt.ylabel("Count")
plt.title("One2Car Listing Price Distribution")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "price_distribution.png", dpi=180)
plt.show()

print("\nDATA PREPARATION COMPLETE")
print("Cleaned:", CLEANED_DIR / "one2car_cleaned.csv")
print("Train:", PROCESSED_DIR / "train.csv")
print("Validation:", PROCESSED_DIR / "validation.csv")
print("Test:", PROCESSED_DIR / "test.csv")
